# Round mosaics — live quick look at every round

Builds one mosaic per color for the rounds in `SELECTED_ROUNDS`, from one
frame per FOV near `TARGET_Z_UM`. Light enough to run during acquisition,
straight off the NAS. Flat-field correction is off by default (`ENABLE_FFC`),
and the 488 nm bead channel is left out (`EXCLUDED_COLORS`). A quick look,
not `round_scheduler.ipynb`'s mosaics; it saves to `SAMPLE_DIR/figures/`, so
both can run at once.

Section 6 builds a table of each round's imaged vs. processed FOVs; section 7
processes what is outstanding:
- one round + `LIVE_LOOP = False`: build it once from what is imaged now.
- `"all"` + `LIVE_LOOP = False`: one pass over every finished round.
- `"all"` + `LIVE_LOOP = True` (default): keep going until every round is
  imaged and processed; start once, leave it running.

Any mix works, e.g. `SELECTED_ROUNDS = ["cells", 6]` with `LIVE_LOOP = True`.

Runs serially: each FOV is one frame plus a thumbnail, too little work for
SLURM's overhead.

Reads share the disk/network with HAL's writes; `CATCHUP_READ_DELAY_SEC`
paces them.

## 1 — Setup

In [ ]:
import os
import sys
import time
from pathlib import Path

from IPython.display import display, clear_output

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/during_imaging/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.live_round_mosaic       import LiveRoundMosaicBuilder, register_focustest_round
from MERci.plots.round_mosaic_plots import show_round_mosaic
from MERci.plots.experiment_plots import get_merci_figures_dir

print(f"SAMPLE_DIR: {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX  = ".zarr"   # same as HAL writes
NOTEBOOK_NAME = "round_mosaics"   # cache and figures folder name

# Camera-to-stage orientation (MERlin microscope YAML), applied to every frame.
MICROSCOPE = "MF3"

# Stage z (um) of the mosaics: per round/color, the frame closest to it. Re-run
# section 3 after changing it.
TARGET_Z_UM = 10.0

# Colors left out: by default 488 nm (beads, always at the bead z). [] includes
# every color, e.g. for a bead mosaic of one round.
EXCLUDED_COLORS = [488.0]

# "all", or a list of labels ("cells", by imaging_type) and/or imaging_round
# numbers, e.g. ["cells", 6]. Processed in that order.
SELECTED_ROUNDS = "all"

# True: keep going until every selected round is imaged and processed.
# False: process what is there now, once.
LIVE_LOOP = True

# Flat-field correction: off by default. On: FFC_N_FOVS extra reads per color, once (cached).
ENABLE_FFC = False
FFC_N_FOVS = 10

# One contrast range per round/color, from CONTRAST_SAMPLE_N_FOVS random interior
# FOVs (more likely to have tissue), so tiles match before the round is done.
CONTRAST_SAMPLE_N_FOVS = 10
CONTRAST_LOW_PCT  = 5.0
CONTRAST_HIGH_PCT = 99.0

POLL_INTERVAL_SEC = 5         # must be well under the time to acquire one FOV
MAX_RUNTIME_MIN    = 24*60*7  # stop after this long (a whole experiment)

# Minimum seconds between on-screen redraws (tiles are placed immediately regardless).
LIVE_REDRAW_MIN_INTERVAL_SEC = 0.5

# Long-side size (px) of the on-screen preview only; the saved PNG is full size.
# Drawing a full canvas (~10600x6700 px) takes ~6 s; downsampled, < 0.1 s.
LIVE_PREVIEW_MAX_PX = 1400

# Seconds between rewrites of the full-size PNG while a round builds (~3 s
# each). Always rewritten once more at the end.
DISK_SAVE_MIN_INTERVAL_SEC = 20.0

# Gap between fresh FOV reads in a catch-up burst (shared link with HAL's writes).
CATCHUP_READ_DELAY_SEC = 0.02

config = ExperimentConfig.from_sample_dir(
    SAMPLE_DIR,
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
    microscope     = MICROSCOPE,
)
meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt,
                                config.data_dir, image_suffix=config.image_suffix)

# The focus-lock test, if present, is added as round_id=0.
from MERci.common.metadata import FOCUSTEST_ROUND_ID
INCLUDE_FOCUSTEST_ROUND = True   # False to leave it out

if INCLUDE_FOCUSTEST_ROUND:
    if register_focustest_round(meta, config, MICROSCOPE, FOCUSTEST_ROUND_ID):
        print(f"Registered focus-test round (round_id={FOCUSTEST_ROUND_ID}).")
    else:
        print(f"No focus-test HAL config found for {MICROSCOPE} (or round "
              f"{FOCUSTEST_ROUND_ID} already registered) -- skipping.")

THUMBNAILS_DIR = config.analysis_dir / "thumbnails"   # same as fov_scheduler
THUMBNAILS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = config.analysis_dir / "cache" / NOTEBOOK_NAME   # FFC fields only
CACHE_DIR.mkdir(parents=True, exist_ok=True)
# figures/, not analysis/mosaics/, which round_scheduler uses.
FIGURES_DIR = get_merci_figures_dir(SAMPLE_DIR, "during_imaging", NOTEBOOK_NAME)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Sample name  : {SAMPLE_NAME}")
print(f"MICROSCOPE   : {MICROSCOPE}")
print(f"FOVs (total) : {meta.n_fovs}")
print(f"Rounds       : {sorted(meta.rounds)}")
print(f"TARGET_Z_UM  : {TARGET_Z_UM}")
print(f"ENABLE_FFC   : {ENABLE_FFC}")
print(f"LIVE_LOOP    : {LIVE_LOOP}")

MICROSCOPE_ORIENTATION_DIR = MERCI_DIR / "data" / "configs" / "merlin" / "microscope"
MOSAIC_BUILDER = LiveRoundMosaicBuilder(
    config=config, metadata=meta,
    microscope=MICROSCOPE, microscope_orientation_dir=MICROSCOPE_ORIENTATION_DIR,
    cache_dir=CACHE_DIR, figures_dir=FIGURES_DIR, thumbnails_dir=THUMBNAILS_DIR,
    notebook_name=NOTEBOOK_NAME,
    target_z_um=TARGET_Z_UM, excluded_colors=EXCLUDED_COLORS,
    focustest_round_id=FOCUSTEST_ROUND_ID,
    enable_ffc=ENABLE_FFC, ffc_n_fovs=FFC_N_FOVS,
    contrast_sample_n_fovs=CONTRAST_SAMPLE_N_FOVS,
    contrast_low_pct=CONTRAST_LOW_PCT, contrast_high_pct=CONTRAST_HIGH_PCT,
    live_preview_max_px=LIVE_PREVIEW_MAX_PX,
    live_redraw_min_interval_sec=LIVE_REDRAW_MIN_INTERVAL_SEC,
    disk_save_min_interval_sec=DISK_SAVE_MIN_INTERVAL_SEC,
    catchup_read_delay_sec=CATCHUP_READ_DELAY_SEC,
)

## 3 — Resolve each round's real colors -> nearest-z frame index

For each round's frame table (via its HAL config) and each color not in
`EXCLUDED_COLORS`, the frame closest to `TARGET_Z_UM`.

In [ ]:
ROUND_COLOR_FRAMES = {}
for round_id in sorted(meta.rounds):
    cf = MOSAIC_BUILDER.resolve_round_color_frames(round_id)
    if cf:
        ROUND_COLOR_FRAMES[round_id] = cf
    print(f"Round {round_id}: colors {sorted(cf)} -> frame indices {cf}")

## 4 — Resolve round labels and `SELECTED_ROUNDS`

A round's label is `"cells"` if one of its series is `imaging_type=="cells"`,
else its `imaging_round` number.

In [ ]:
if SELECTED_ROUNDS == "all":
    SELECTED_ROUND_IDS = sorted(ROUND_COLOR_FRAMES)
else:
    # A bare string ("cells") is one round, not a list of letters.
    selected_tokens = [SELECTED_ROUNDS] if isinstance(SELECTED_ROUNDS, str) else SELECTED_ROUNDS
    SELECTED_ROUND_IDS = [MOSAIC_BUILDER.resolve_round_token(t) for t in selected_tokens]
    unresolved = [r for r in SELECTED_ROUND_IDS if r not in ROUND_COLOR_FRAMES]
    if unresolved:
        raise ValueError(f"Round id(s) {unresolved} have no resolved colors "
                          f"(see section 3) -- check round_info.csv.")

print(f"SELECTED_ROUND_IDS: {SELECTED_ROUND_IDS} "
      f"(labels: {[MOSAIC_BUILDER.round_label_for(r) for r in SELECTED_ROUND_IDS]})")

## 5 — Mosaic builder stats

`MOSAIC_BUILDER` (`LiveRoundMosaicBuilder`, section 2) keeps FFC fields,
contrast ranges, tile geometry and canvases between polls, and does the
reading and placing. Its docstrings explain the design.

In [ ]:
print(f"Microscope orientation ({MICROSCOPE}): {MOSAIC_BUILDER.microscope_orientation}")
print(f"STEP_SIZE_UM       : {MOSAIC_BUILDER.step_size_um:.1f} um")
print(f"Exterior FOV count : {len(MOSAIC_BUILDER.exterior_fov_ids)}")

## 6 — Build the state table

Per selected round: FOVs imaged, FOVs processed (thumbnail for every color),
total FOVs. Section 7 rebuilds it every poll; this is a quick look.

In [ ]:
STATE_DF = MOSAIC_BUILDER.build_state_df(SELECTED_ROUND_IDS, ROUND_COLOR_FRAMES)
display(STATE_DF)

## 7 — Process selected rounds (one loop for all three use cases)

Every cycle, processes rounds with `processed_fovs < imaged_fovs` in
`SELECTED_ROUND_IDS` order. The figure updates tile by tile (at most every
`LIVE_REDRAW_MIN_INTERVAL_SEC`). With `LIVE_LOOP`, runs until every round is
imaged and processed (or `MAX_RUNTIME_MIN`). Interrupt the kernel to stop;
what was built is saved to `figures/`.

In [ ]:
start_time = time.time()
poll_count = 0
last_processed_round_id = None   # last round built, redrawn on idle cycles

try:
    while True:
        if (time.time() - start_time) > MAX_RUNTIME_MIN * 60:
            print(f"Stopping: MAX_RUNTIME_MIN={MAX_RUNTIME_MIN} exceeded.")
            break

        poll_count += 1
        state_df = MOSAIC_BUILDER.build_state_df(SELECTED_ROUND_IDS, ROUND_COLOR_FRAMES)
        fully_done = True
        processed_anything = False

        for _, row in state_df.iterrows():
            round_id = row["round_id"]
            if row["imaged_fovs"] < row["total_fovs"]:
                fully_done = False   # still imaging: keep watching
            if row["processed_fovs"] < row["imaged_fovs"]:
                fully_done = False
                processed_anything = True
                color_frames = ROUND_COLOR_FRAMES[round_id]
                fov_ids = MOSAIC_BUILDER.round_imaged_fov_ids(round_id)
                print(f"poll #{poll_count} | round {row['round']}: processing "
                      f"{row['processed_fovs']} -> {len(fov_ids)}/{row['total_fovs']} FOVs...")
                MOSAIC_BUILDER.build_round_mosaic(round_id, color_frames, fov_ids, row["round"])
                last_processed_round_id = round_id

        # Refresh, so the counts include what this cycle built.
        state_df = MOSAIC_BUILDER.build_state_df(SELECTED_ROUND_IDS, ROUND_COLOR_FRAMES)
        elapsed = time.time() - start_time

        # Idle cycle: clear the output (or status lines pile up) and redraw the
        # last canvas (no new reads), so the mosaic stays on screen.
        if not processed_anything:
            if last_processed_round_id is not None:
                last_color_frames = ROUND_COLOR_FRAMES[last_processed_round_id]
                show_round_mosaic(
                    last_processed_round_id,
                    {c: MOSAIC_BUILDER.get_canvas(last_processed_round_id, c) for c in last_color_frames},
                    MOSAIC_BUILDER.round_label_for(last_processed_round_id),
                    mosaic_paths={c: MOSAIC_BUILDER.mosaic_path(last_processed_round_id, c)
                                  for c in last_color_frames},
                    live_preview_max_px=LIVE_PREVIEW_MAX_PX, save_full_res=False,
                )
            else:
                clear_output(wait=True)
        print(f"poll #{poll_count} | elapsed {elapsed / 60:.1f} min | next check in {POLL_INTERVAL_SEC}s")
        print(state_df.to_string(index=False))

        if not LIVE_LOOP:
            print("LIVE_LOOP is False -- one pass complete.")
            break
        if fully_done:
            print("All selected rounds fully imaged and processed.")
            break

        time.sleep(POLL_INTERVAL_SEC)
except KeyboardInterrupt:
    print(f"Stopped by user after poll #{poll_count}.")